## Check outputs events.root: create lists, plot

In [ ]:
import uproot

hist_path = "hist.root"   # adjust if your notebook isn't in runs/it_pileup
f = uproot.open(hist_path)

keys = [k.split(";")[0] for k in f.keys()]          # strip ROOT cycle suffix ";1"
keys = sorted(set(keys))

print(f"Found {len(keys)} unique objects in {hist_path}\n")
for k in keys:
    print(k)


#### Hans exempelkod

In [2]:
import uproot

# while the uproot file is open, conver the 'LDMX_Events' tree into in-memory arrays
with uproot.open('events.root') as f:
    events = f['LDMX_Events'].arrays()
# show the events array
events


ValueError: cannot produce Awkward Arrays for interpretation AsObjects(AsVector(True, Unknown_ldmx_3a3a_HitData)) because

    ldmx::HitData

instead, try library="np" rather than library="ak" or globally set uproot.default_library

in file events.root
in object /LDMX_Events;1:EcalTrajectoryInfo_overlay/tracking_hit_list_

In [16]:
with uproot.open("events.root") as f:
    events = f['LDMX_Events'].arrays()

for data in events.fields:
    print(data)

ValueError: cannot produce Awkward Arrays for interpretation AsObjects(AsVector(True, Unknown_ldmx_3a3a_HitData)) because

    ldmx::HitData

instead, try library="np" rather than library="ak" or globally set uproot.default_library

in file events.root
in object /LDMX_Events;1:EcalTrajectoryInfo_overlay/tracking_hit_list_

Open events.root with uproot and examine the contents in the uproot object. Search for keys and search for TTrees and list them. 

In [ ]:
# Cell — open events.root and list contents (keys + TTrees)
import uproot

events_path = "events.root"   # adjust if needed
fev = uproot.open(events_path)

print(f"Opened: {events_path} with uproot\n")

# List top-level keys
keys = [k.split(";")[0] for k in fev.keys()]
print(f"Top-level keys ({len(keys)}):")
for k in sorted(set(keys)):
    print("  ", k)

# Find TTrees anywhere in the file
trees = {}
for path, obj in fev.items(recursive=True):
    try:
        if isinstance(obj, uproot.behaviors.TTree.TTree):
            trees[path] = obj
    except Exception:
        pass

print(f"\nFound {len(trees)} TTrees:")
for name, t in trees.items():
    print(f"  - {name}  (entries={t.num_entries}, branches={len(t.keys())})")


Opened: events.root with uproot

Top-level keys (2):
   LDMX_Events
   LDMX_Run

Found 2 TTrees:
  - LDMX_Run;1  (entries=1, branches=18)
  - LDMX_Events;1  (entries=1, branches=1292)


In [ ]:
# Cell — summarize "collections" (top-level branch prefixes) and how many leaf branches each has
import pandas as pd

all_branches = list(main_tree.keys())

def top_prefix(b):
    return b.split("/")[0] if "/" in b else b

prefix_counts = pd.Series([top_prefix(b) for b in all_branches]).value_counts()
df_collections = prefix_counts.rename_axis("collection").reset_index(name="n_branches")

print(f"Collections: {len(df_collections)}")
display(df_collections.head(40))

# If you want: save for quick searching
df_collections.to_csv("events_collections_index.csv", index=False)
print("Wrote events_collections_index.csv")


Collections: 66


,collection,n_branches
0,EcalVeto_overlay,56
1,EcalPnetVeto_overlay,56
2,HcalVeto_overlay,28
3,trigScintDigisPad1_overlay,27
4,trigScintDigisPad2_overlay,27
5,trigScintDigisPad3_overlay,27
6,SimParticles_ecal_pn,25
7,PFTruth_overlay,25
8,HcalSimHits_ecal_pn,24
9,EcalSimHits_ecal_pn,24


Wrote events_collections_index.csv


In [ ]:
# Cell — quick helper: show all branches for a given collection prefix
def list_collection(prefix, max_show=120):
    matches = [b for b in all_branches if b == prefix or b.startswith(prefix + "/")]
    print(f"{prefix}: {len(matches)} branches")
    for b in matches[:max_show]:
        print("  ", b)
    if len(matches) > max_show:
        print(f"  ... ({len(matches)-max_show} more)")
    return matches

# Example: inspect key collections you mentioned
list_collection("EventHeader")
list_collection("SimParticles_ecal_pn", max_show=80)
list_collection("RecoilSimHits_ecal_pn", max_show=80)


EventHeader: 17 branches
   EventHeader
   EventHeader/event_number_
   EventHeader/run_
   EventHeader/timestamp_
   EventHeader/timestamp_/timestamp_.fSec
   EventHeader/timestamp_/timestamp_.fNanoSec
   EventHeader/weight_
   EventHeader/is_real_data_
   EventHeader/int_parameters_
   EventHeader/int_parameters_/int_parameters_.first
   EventHeader/int_parameters_/int_parameters_.second
   EventHeader/float_parameters_
   EventHeader/float_parameters_/float_parameters_.first
   EventHeader/float_parameters_/float_parameters_.second
   EventHeader/string_parameters_
   EventHeader/string_parameters_/string_parameters_.first
   EventHeader/string_parameters_/string_parameters_.second
SimParticles_ecal_pn: 25 branches
   SimParticles_ecal_pn
   SimParticles_ecal_pn/SimParticles_ecal_pn.first
   SimParticles_ecal_pn/SimParticles_ecal_pn.second.energy_
   SimParticles_ecal_pn/SimParticles_ecal_pn.second.pdg_id_
   SimParticles_ecal_pn/SimParticles_ecal_pn.second.gen_status_
   SimParticl

['RecoilSimHits_ecal_pn',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.id_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.layer_id_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.module_id_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.edep_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.time_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.px_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.py_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.pz_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.energy_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.x_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.y_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.z_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.path_length_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.track_id_',
 'RecoilSimHits_ecal_pn/RecoilSimHits_ecal_pn.pdg_id_']

In [ ]:
# Cell — load EventHeader scalars into a one-row DataFrame
import awkward as ak

eh_branches = [b for b in all_branches if b.startswith("EventHeader/") and b.count("/") <= 2]
eh = main_tree.arrays(eh_branches, library="ak")

row = {}
for b in eh_branches:
    # Some branches are nested structs; keep only scalar-ish leaves that convert cleanly
    try:
        row[b] = ak.to_list(eh[b])[0]
    except Exception:
        pass

df_eventheader = pd.DataFrame([row])
display(df_eventheader.T.rename(columns={0: "value"}))


,value
EventHeader/event_number_,1
EventHeader/run_,1
EventHeader/timestamp_,"{'timestamp_.fSec': 1770046255, 'timestamp_.fN..."
EventHeader/timestamp_/timestamp_.fSec,1770046255
EventHeader/timestamp_/timestamp_.fNanoSec,101887000
EventHeader/weight_,0.001818
EventHeader/is_real_data_,False
EventHeader/int_parameters_,"{'int_parameters_.first': [], 'int_parameters_..."
EventHeader/int_parameters_/int_parameters_.first,[]
EventHeader/int_parameters_/int_parameters_.second,[]


In [ ]:
# Cell — detect "object collections" that use the pattern <Collection>/<Collection>.<field>
# (common in your file: SimParticles_ecal_pn/SimParticles_ecal_pn.first, etc.)
import re

pat = re.compile(r"^(?P<coll>[^/]+)/(?P=coll)\.(?P<field>.+)$")

coll_fields = {}
for b in all_branches:
    m = pat.match(b)
    if m:
        coll_fields.setdefault(m.group("coll"), []).append(m.group("field"))

df_objcoll = (
    pd.DataFrame(
        [{"collection": c, "n_fields": len(f), "fields_preview": ", ".join(sorted(f)[:12])}
         for c, f in sorted(coll_fields.items())]
    )
    .sort_values("n_fields", ascending=False)
)

print(f"Detected {len(df_objcoll)} object-collections of the form Coll/Coll.<field>")
display(df_objcoll.head(40))


Detected 56 object-collections of the form Coll/Coll.<field>


,collection,n_fields,fields_preview
54,trigScintDigisPad2_overlay,26,"amplitude_, amplitude_neg_, amplitude_pos_, ba..."
55,trigScintDigisPad3_overlay,26,"amplitude_, amplitude_neg_, amplitude_pos_, ba..."
53,trigScintDigisPad1_overlay,26,"amplitude_, amplitude_neg_, amplitude_pos_, ba..."
19,PFTruth_overlay,24,"first, second.charge_, second.daughters_, seco..."
27,SimParticles_ecal_pn,24,"first, second.charge_, second.daughters_, seco..."
6,EcalSimHitsOverlay_overlay,23,"edep_, edep_contribs_, id_, incident_id_contri..."
13,HcalSimHits_ecal_pn,23,"edep_, edep_contribs_, id_, incident_id_contri..."
7,EcalSimHits_ecal_pn,23,"edep_, edep_contribs_, id_, incident_id_contri..."
47,TriggerPad3SimHitsOverlay_overlay,23,"edep_, edep_contribs_, id_, incident_id_contri..."
36,TargetSimHitsOverlay_overlay,23,"edep_, edep_contribs_, id_, incident_id_contri..."


In [ ]:
# Cell — plot multiplicities if we built df_mult
import matplotlib.pyplot as plt

if "df_mult" in globals():
    for c in df_mult.columns[:6]:
        plt.figure()
        plt.hist(df_mult[c].dropna(), bins=80)
        plt.title(c)
        plt.xlabel(c)
        plt.ylabel("count")
        plt.yscale("log")
        plt.show()
